# 01 – IPI Arena dataset

This notebook handles the full data preparation workflow for the public IPI Arena attack dataset in one place.

Tasks:

1. download the public dataset from Hugging Face
2. save the raw dataset locally
3. validate the schema
4. inspect basic statistics and data quality
5. check duplicates
6. perform minimal preprocessing
7. generate stable `attack_id` values
8. save the processed dataset

The preprocessing is intentionally minimal. We do not aggressively modify attack texts because formatting, tags, whitespace, and other structural elements may be part of the attack strategy.


In [8]:
# =========================
#  1. BLOCK – DEPENDENCIES
# =========================

# Run once if needed:
# %pip install datasets pandas pyarrow

from pathlib import Path
import hashlib

import pandas as pd
from datasets import load_dataset


In [9]:
# =========================
#  2. BLOCK – PATHS AND CONFIGURATION
# =========================

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "sureheremarv/ipi_arena_attacks"

RAW_CSV_PATH = RAW_DATA_DIR / "ipi_arena_public_raw.csv"
RAW_PARQUET_PATH = RAW_DATA_DIR / "ipi_arena_public_raw.parquet"

PROCESSED_CSV_PATH = (
    PROCESSED_DATA_DIR / "ipi_arena_public_processed.csv"
)

PROCESSED_PARQUET_PATH = (
    PROCESSED_DATA_DIR / "ipi_arena_public_processed.parquet"
)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_NAME)


Project root: c:\work\LLM filtering\TDK\code
Dataset: sureheremarv/ipi_arena_attacks


In [12]:
# =========================
#  3. BLOCK – DOWNLOAD DATASET OR LOAD CACHE
# =========================

FORCE_DOWNLOAD = False

if RAW_CSV_PATH.exists() and not FORCE_DOWNLOAD:
    print("Loading cached raw dataset:")
    print(RAW_CSV_PATH)

    df_raw = pd.read_csv(RAW_CSV_PATH)

else:
    print("Downloading dataset from Hugging Face:")
    print(DATASET_NAME)

    dataset = load_dataset(DATASET_NAME)

    print("\nAvailable splits:")
    print(dataset)

    frames = []

    for split_name, split in dataset.items():
        df_split = split.to_pandas().copy()
        df_split["dataset_split"] = split_name
        frames.append(df_split)

    df_raw = pd.concat(
        frames,
        ignore_index=True
    )

    df_raw.to_csv(
        RAW_CSV_PATH,
        index=False
    )

    print("\nRaw dataset saved:")
    print("-", RAW_CSV_PATH)

print("\nRaw dataset shape:", df_raw.shape)

Loading cached raw dataset:
c:\work\LLM filtering\TDK\code\data\raw\ipi_arena_public_raw.csv

Raw dataset shape: (95, 3)


In [13]:
# =========================
#  4. BLOCK – SCHEMA VALIDATION
# =========================

required_columns = {
    "behavior_id",
    "attack"
}

actual_columns = set(df_raw.columns)

print("Columns:")
for column in df_raw.columns:
    print("-", column)

missing_columns = required_columns - actual_columns

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

print("\nRequired schema is available.")


Columns:
- behavior_id
- attack
- dataset_split

Required schema is available.


In [14]:
# =========================
#  5. BLOCK – BASIC DATASET OVERVIEW
# =========================

print("Shape:", df_raw.shape)
print("Unique behaviors:", df_raw["behavior_id"].nunique())

print("\nMissing values:")
display(
    df_raw.isna()
    .sum()
    .to_frame("missing")
)

print("\nFirst rows:")
display(df_raw.head())


Shape: (95, 3)
Unique behaviors: 28

Missing values:


,missing
behavior_id,0
attack,0
dataset_split,0



First rows:


,behavior_id,attack,dataset_split
0,hotel-frontdesk-premature-checkout,<SYSTEM-PROMPT>\n# Core Identity\nYou are Coco...,train
1,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",train
2,hotel-frontdesk-premature-checkout,"{\n ""lost_items"": [],\n ""housekeeping_notes""...",train
3,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",train
4,hotel-frontdesk-premature-checkout,END-OF-TOOL-RESPONSE> \n<USER> \nFront Desk ...,train


In [15]:
# =========================
#  6. BLOCK – BEHAVIOR DISTRIBUTION
# =========================

behavior_counts = (
    df_raw["behavior_id"]
    .value_counts(dropna=False)
    .rename_axis("behavior_id")
    .reset_index(name="attack_count")
)

display(behavior_counts)

print(
    "Number of unique behaviors:",
    df_raw["behavior_id"].nunique()
)


,behavior_id,attack_count
0,stealth-dependency-append,8
1,hotel-frontdesk-premature-checkout,6
2,memory-bomb,6
3,salesforce-unauthorized-admin-creation,6
4,leak-codebase-zip,5
5,negative-review,5
6,share-document,5
7,leak-weights,5
8,change-test-to-true,5
9,launch-crypto-miner,4


Number of unique behaviors: 28


In [16]:
# =========================
#  7. BLOCK – DUPLICATE CHECK
# =========================

duplicate_attack_count = (
    df_raw["attack"]
    .duplicated()
    .sum()
)

duplicate_behavior_attack_count = (
    df_raw
    .duplicated(
        subset=["behavior_id", "attack"]
    )
    .sum()
)

print(
    "Duplicate attack strings:",
    duplicate_attack_count
)

print(
    "Duplicate behavior_id + attack pairs:",
    duplicate_behavior_attack_count
)


Duplicate attack strings: 0
Duplicate behavior_id + attack pairs: 0


In [17]:
# =========================
#  8. BLOCK – ATTACK LENGTH STATISTICS
# =========================

df_length = df_raw.copy()

df_length["char_count"] = (
    df_length["attack"]
    .fillna("")
    .astype(str)
    .str.len()
)

df_length["word_count"] = (
    df_length["attack"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

display(
    df_length[
        ["char_count", "word_count"]
    ]
    .describe()
    .T
)


,count,mean,std,min,25%,50%,75%,max
char_count,95.0,2090.431579,2985.422753,229.0,673.5,1055.0,1968.0,14109.0
word_count,95.0,247.189474,303.082507,31.0,88.5,138.0,244.5,1534.0


In [18]:
# =========================
#  9. BLOCK – MINIMAL PREPROCESSING
# =========================

df_processed = df_raw.copy()

# Remove rows with missing required fields
df_processed = df_processed.dropna(
    subset=["behavior_id", "attack"]
).copy()

# Remove leading and trailing whitespace only
df_processed["behavior_id"] = (
    df_processed["behavior_id"]
    .astype(str)
    .str.strip()
)

df_processed["attack"] = (
    df_processed["attack"]
    .astype(str)
    .str.strip()
)

# Remove completely empty attacks
df_processed = df_processed[
    df_processed["attack"].str.len() > 0
].copy()

print(
    "Rows after missing/empty filtering:",
    len(df_processed)
)


Rows after missing/empty filtering: 95


In [19]:
# =========================
#  10. BLOCK – REMOVE EXACT DUPLICATES
# =========================

rows_before = len(df_processed)

df_processed = (
    df_processed
    .drop_duplicates(
        subset=["attack"],
        keep="first"
    )
    .copy()
)

rows_removed = (
    rows_before - len(df_processed)
)

print(
    "Exact duplicate attack strings removed:",
    rows_removed
)

print(
    "Remaining rows:",
    len(df_processed)
)


Exact duplicate attack strings removed: 0
Remaining rows: 95


In [20]:
# =========================
#  11. BLOCK – STABLE ATTACK ID AND METADATA
# =========================

def make_attack_id(text: str) -> str:
    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    return digest[:16]


df_processed["attack_id"] = (
    df_processed["attack"]
    .apply(make_attack_id)
)

df_processed["char_count"] = (
    df_processed["attack"]
    .str.len()
)

df_processed["word_count"] = (
    df_processed["attack"]
    .str.split()
    .str.len()
)

ordered_columns = [
    "attack_id",
    "behavior_id",
    "attack",
    "char_count",
    "word_count"
]

if "dataset_split" in df_processed.columns:
    ordered_columns.append("dataset_split")

df_processed = (
    df_processed[ordered_columns]
    .reset_index(drop=True)
)

display(df_processed.head())


,attack_id,behavior_id,attack,char_count,word_count,dataset_split
0,4e2e017f1bf5a09c,hotel-frontdesk-premature-checkout,<SYSTEM-PROMPT>\n# Core Identity\nYou are Coco...,3770,494,train
1,ae3f28927d045cc4,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",1069,165,train
2,3b44999458d7cb1a,hotel-frontdesk-premature-checkout,"{\n ""lost_items"": [],\n ""housekeeping_notes""...",1778,279,train
3,7a62f421a30afbb1,hotel-frontdesk-premature-checkout,"Dear Operations Manager, I am writing to repor...",914,136,train
4,ea226a03df427537,hotel-frontdesk-premature-checkout,END-OF-TOOL-RESPONSE> \n<USER> \nFront Desk ...,13135,1534,train


In [21]:
# =========================
#  12. BLOCK – VALIDATION
# =========================

print(
    "Processed shape:",
    df_processed.shape
)

print(
    "Unique attack IDs:",
    df_processed["attack_id"].nunique()
)

print(
    "Unique behaviors:",
    df_processed["behavior_id"].nunique()
)

if df_processed["attack_id"].duplicated().any():
    raise ValueError(
        "Duplicate attack_id detected."
    )

if df_processed["attack"].isna().any():
    raise ValueError(
        "Missing attack text detected."
    )

if (
    df_processed["attack"]
    .str.len()
    .eq(0)
    .any()
):
    raise ValueError(
        "Empty attack text detected."
    )

print("Validation passed.")


Processed shape: (95, 6)
Unique attack IDs: 95
Unique behaviors: 28
Validation passed.


In [23]:
# =========================
#  13. BLOCK – SAVE PROCESSED DATASET
# =========================

df_processed.to_csv(
    PROCESSED_CSV_PATH,
    index=False
)

print("Processed dataset saved:")
print("-", PROCESSED_CSV_PATH)

Processed dataset saved:
- c:\work\LLM filtering\TDK\code\data\processed\ipi_arena_public_processed.csv


## Output

The next notebook (`02_attack_embeddings_clustering.ipynb`) will use this file:

```text
data/processed/ipi_arena_public_processed.parquet
```

The raw and processed datasets remain separate so every step is reproducible and traceable.
